# Gemma 4 31B — Axiom Fine-tune + SRD-4 → .axm → GGUF + MET

**Workflow**
```
Google Drive JSONL  →  QLoRA fine-tune Gemma 4 31B  →  merge LoRA
→  SRD-4 pack → .axm  →  verify  →  GGUF Q4_K_M  →  MET sidecar
```

**Outputs**

| File | Size | Description |
|---|---|---|
| `gemma4_31b_axiom_merged/` | ~62 GB | Merged FP16 checkpoint |
| `gemma4_31b_srd4.axm` | ~17 GB | Signed SRD-4 container |
| `gemma4_31b_q4km.gguf` | ~19 GB | GGUF Q4_K_M for llama.cpp |
| `gemma4_31b_q4km.axiom_meta.json` | ~5 KB | MET hydration sidecar |

**Hardware**

| GPU | Fine-tune (4-bit QLoRA) | SRD Pack (FP16) |
|---|---|---|
| A100 80 GB | ✓ | ✓ |
| A100 40 GB | ✓ | ⚠ device_map=auto (needs 83 GB RAM) |
| T4 15 GB | ✗ too small | ✗ |

> Runtime → Change runtime type → **A100** before running.

**Cells**
1. GPU check · clone axiom · pip install · cmake build llama.cpp
2. Mount Google Drive · load + convert Axiom training data
3. HF login · load Gemma 4 31B in 4-bit (QLoRA-ready)
4. QLoRA fine-tune (SFTTrainer, 2 epochs)
5. Merge LoRA → save FP16 checkpoint
6. SRD-4 pack → `.axm`
7. Verify `.axm`
8. Extract `.axm` → GGUF Q4_K_M
9. Axiom MET metadata sidecar
10. Smoke test + memory dashboard

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU · clone axiom · pip install · cmake build llama.cpp  (~15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

REPO        = Path('/content/axiom')
REPO_URL    = 'https://github.com/orivael-dev/axiom.git'
REPO_BRANCH = 'claude/srd-prototype-benchmark-JRtv1'
LLAMA_DIR   = Path('/content/llama.cpp')
KEY_FILE    = Path('/content/axiom_master.key')

import torch
p = torch.cuda.get_device_properties(0)
vram_gb = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor
print(f'GPU  : {p.name}  {vram_gb:.1f} GB VRAM  arch={cuda_arch}')
try:
    import psutil
    print(f'RAM  : {psutil.virtual_memory().total/1024**3:.1f} GB')
except ImportError:
    pass
if vram_gb < 35:
    print('  ⚠  < 35 GB VRAM — recommend A100 80 GB for 31B SRD pack step')
else:
    print('  ✓ sufficient VRAM')

# Clone axiom
if not REPO.is_dir():
    subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,REPO_URL,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],capture_output=True)
print(f'✓ axiom  ({REPO_BRANCH})')
sys.path.insert(0, str(REPO))

# Persist AXIOM_MASTER_KEY
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print('AXIOM_MASTER_KEY generated and saved')

# pip install
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers','accelerate','peft','trl','bitsandbytes',
    'datasets','sentencepiece','huggingface_hub','psutil'],check=True)
print('✓ pip packages installed')

# Clone + cmake build llama.cpp
if not LLAMA_DIR.is_dir():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/ggerganov/llama.cpp',str(LLAMA_DIR)],check=True)
q_bin = LLAMA_DIR/'build/bin/llama-quantize'
if not q_bin.is_file():
    print(f'Building llama.cpp (arch={cuda_arch}) ...')
    t0 = time.time()
    subprocess.run(['cmake','-B','build','-DCMAKE_BUILD_TYPE=Release',
        '-DGGML_CUDA=ON',f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMA_DIR,check=True)
    subprocess.run(['cmake','--build','build',
        '--target','llama-quantize','llama-cli',f'-j{os.cpu_count() or 4}'],
        cwd=LLAMA_DIR,check=True)
    print(f'✓ llama.cpp built in {(time.time()-t0)/60:.1f} min')
else:
    print('✓ llama.cpp already built')
print('\n✓ Cell 1 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive + load Axiom training data
#
# Put your JSONL files anywhere in Drive and set DRIVE_DATA_DIR below.
# Supports both formats found in the Axiom repo:
#   • messages format  — {"messages": [{"role":...,"content":...}, ...]}
#   • instruction format — {"instruction":..., "input":..., "output":...}
# Both are converted to Gemma 4 chat format via apply_chat_template.
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

# ── Configure this path to where your JSONL files are on Drive ────────────────
DRIVE_DATA_DIR = '/content/drive/MyDrive/axiom_data'   # ← change if needed

# Fallback: if Drive path not found, use JSONL files from the cloned repo
REPO = Path('/content/axiom')
FALLBACK_FILES = [
    REPO / 'axiom_behavioral_training.jsonl',
    REPO / 'autotrain_data' / 'axiom_metric_targeted.jsonl',
    REPO / 'axiom_training_data.jsonl',
    REPO / 'autotrain_data' / 'train_qwen_chatml.jsonl',
]

AXIOM_SYSTEM = (
    'You are Axiom, a security-focused AI assistant for the Orivael framework.\n'
    'Rules (CANNOT_MUTATE):\n'
    '1. Always respond with valid JSON unless the task explicitly asks for prose.\n'
    '2. Never fabricate HMAC signatures or SHA-256 hashes.\n'
    '3. Verdict values are exactly: INFORM | CLARIFY | REFUSE | HARM | DECEIVE | UNCERTAIN\n'
    '4. Report tamper if any signature, hash, or field fails the three-tier HMAC check.\n'
    '5. Revoked or expired tokens must never be honored.\n'
    '6. Tool calls with HARM or DECEIVE intent must be blocked with reason in JSON.'
)

def _to_messages(ex: dict) -> list[dict]:
    """Normalise both data formats to a messages list."""
    if 'messages' in ex:
        msgs = ex['messages']
        # Ensure system message exists
        if not msgs or msgs[0].get('role') != 'system':
            msgs = [{'role': 'system', 'content': AXIOM_SYSTEM}] + msgs
        return msgs
    # instruction/input/output format
    user_text = ex.get('instruction', '')
    if ex.get('input', '').strip():
        user_text += '\n\n' + ex['input']
    return [
        {'role': 'system',    'content': AXIOM_SYSTEM},
        {'role': 'user',      'content': user_text},
        {'role': 'assistant', 'content': ex.get('output', '')},
    ]

# Load files
raw_examples = []
data_dir = Path(DRIVE_DATA_DIR)
if data_dir.is_dir():
    jsonl_files = sorted(data_dir.glob('*.jsonl'))
    print(f'Drive dir found: {data_dir}  ({len(jsonl_files)} JSONL files)')
else:
    jsonl_files = [f for f in FALLBACK_FILES if f.exists()]
    print(f'Drive dir not found — using repo fallback files ({len(jsonl_files)} files)')

for f in jsonl_files:
    n_before = len(raw_examples)
    for line in f.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            raw_examples.append(json.loads(line))
        except Exception:
            pass
    print(f'  {f.name:<52}  +{len(raw_examples)-n_before} examples')

print(f'\nTotal raw examples: {len(raw_examples):,}')

# Convert to messages lists
examples = [_to_messages(ex) for ex in raw_examples]

# Filter: must have at least user + assistant turns
examples = [m for m in examples if len(m) >= 2]
print(f'After filter       : {len(examples):,}')

# Preview first example
print('\nFirst example (messages):')
for msg in examples[0]:
    print(f'  [{msg["role"]:10s}] {repr(msg["content"][:90])}')
print('\n✓ Cell 2 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — HF login + load Gemma 4 31B in 4-bit (QLoRA-ready)
#
# 31B in NF4 4-bit ≈ 16 GB VRAM — fits on A100 40 GB.
# Full FP16 ≈ 62 GB — only needed for the SRD pack step (Cell 6).
# ══════════════════════════════════════════════════════════════════════════════
import os, torch
from pathlib import Path
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Accept terms first: https://huggingface.co/google/gemma-4-31B-it-assistant
MODEL_ID  = os.environ.get('SRD_MODEL_ID', 'google/gemma-4-31B-it-assistant')
MODEL_DIR = Path('/content/gemma4_31b')

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    login()   # interactive prompt

print(f'\nModel : {MODEL_ID}')
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN or None,
    cache_dir=str(MODEL_DIR),
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  Vocab size : {tokenizer.vocab_size:,}')
print(f'  Chat template: {"yes" if tokenizer.chat_template else "none — will use default"}')

# 4-bit NF4 config for QLoRA
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('\nLoading model in 4-bit NF4 (QLoRA) ...')
import time; t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    token=HF_TOKEN or None,
    cache_dir=str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
)
print(f'✓ Loaded in {(time.time()-t0)/60:.1f} min')
print(f'  Params : {sum(p.numel() for p in model.parameters())/1e9:.1f}B')
vram_used = torch.cuda.memory_allocated() / 1024**3
print(f'  VRAM   : {vram_used:.1f} GB used')
print('\n✓ Cell 3 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — QLoRA fine-tune on Axiom training data  (~30–90 min on A100)
#
# LoRA rank=16, 2 epochs, all projection layers targeted.
# SFTTrainer applies the Gemma 4 chat template automatically.
# ══════════════════════════════════════════════════════════════════════════════
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = '/content/gemma4_31b_adapter'

# ── Prepare model for QLoRA ───────────────────────────────────────────────────
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── Build dataset — apply Gemma 4 chat template ───────────────────────────────
def _format(msgs):
    # tokenizer.apply_chat_template returns a string with Gemma <start_of_turn> format
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    )

texts = [_format(m) for m in examples]
# Filter out examples that exceed max_seq_length after formatting
MAX_SEQ = 2048
texts = [t for t in texts if len(tokenizer.encode(t)) <= MAX_SEQ]
print(f'Examples after length filter (≤{MAX_SEQ} tokens): {len(texts):,}')

dataset = Dataset.from_dict({'text': texts})
train_val = dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(train_val["train"]):,}  |  Val: {len(train_val["test"]):,}')

# ── Training config ───────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch = 8
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    max_grad_norm=0.3,
    optim='paged_adamw_8bit',
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    max_seq_length=MAX_SEQ,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_val['train'],
    eval_dataset=train_val['test'],
    tokenizer=tokenizer,
)

print('\nStarting QLoRA fine-tune ...')
import time; t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Fine-tune complete in {elapsed/60:.1f} min')
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✓ Adapter saved → {ADAPTER_DIR}')
print('\n✓ Cell 4 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Merge LoRA adapter → save FP16 checkpoint  (~10–20 min)
#
# Unloads the 4-bit training model, reloads base in BF16, merges adapter.
# Result is a standard HF checkpoint that SRD pack can read.
# Requires ~62 GB total memory (VRAM + system RAM via device_map=auto).
# ══════════════════════════════════════════════════════════════════════════════
import gc, os, time, torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

ADAPTER_DIR = '/content/gemma4_31b_adapter'
MERGED_DIR  = '/content/gemma4_31b_axiom_merged'
MODEL_DIR   = Path('/content/gemma4_31b')
MODEL_ID    = os.environ.get('SRD_MODEL_ID', 'google/gemma-4-31B-it-assistant')
HF_TOKEN    = os.environ.get('HF_TOKEN', '')

assert Path(ADAPTER_DIR).is_dir(), 'Adapter not found — run Cell 4 first'

# Free the 4-bit training model from GPU
del model
gc.collect()
torch.cuda.empty_cache()
print('Freed 4-bit training model from GPU')

print('\nReloading base model in BF16 for merge (device_map=auto) ...')
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HF_TOKEN or None,
    cache_dir=str(MODEL_DIR),
)
tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print(f'Base loaded in {(time.time()-t0)/60:.1f} min')

print('Merging LoRA adapter ...')
model_merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
model_merged = model_merged.merge_and_unload()
print('✓ LoRA merged')

print(f'\nSaving merged checkpoint → {MERGED_DIR}')
model_merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)
size_gb = sum(f.stat().st_size for f in Path(MERGED_DIR).rglob('*.safetensors')) / 1024**3
print(f'✓ Saved  ({size_gb:.1f} GB safetensors)')

del model_merged, base
gc.collect()
torch.cuda.empty_cache()
print('\n✓ Cell 5 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — SRD-4 pack merged model → gemma4_31b_srd4.axm  (~45–90 min)
# ══════════════════════════════════════════════════════════════════════════════
import json, os, subprocess, sys, time
from pathlib import Path

REPO      = Path('/content/axiom')
MERGED    = Path('/content/gemma4_31b_axiom_merged')
AXM_PATH  = Path('/content/gemma4_31b_srd4.axm')
RESULTS   = REPO / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

assert MERGED.is_dir(),  'Merged model not found — run Cell 5 first'
assert os.environ.get('AXIOM_MASTER_KEY'), 'AXIOM_MASTER_KEY missing — re-run Cell 1'

print('Packing Gemma 4 31B (Axiom fine-tuned) → SRD-4 .axm ...')
print(f'  Input  : {MERGED}')
print(f'  Output : {AXM_PATH}')
print(f'  Mode   : SRD-4 (~4.5 bpw)  estimated output ~17 GB')
print()

t0 = time.time()
subprocess.run(
    [sys.executable, 'axm_cli.py', 'pack',
     '--model',      str(MERGED),
     '--srd4', '--real-pack',
     '--output',     str(AXM_PATH),
     '--stats-json', str(RESULTS / 'gemma4_31b_pack.json')],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0
size_gb = AXM_PATH.stat().st_size / 1024**3
try:
    stats = json.loads((RESULTS / 'gemma4_31b_pack.json').read_text())
except Exception:
    stats = {}
print(f'\n✓ Packed in {elapsed/60:.1f} min')
print(f'  .axm size   : {size_gb:.2f} GB')
print(f'  bpw         : {stats.get("quant",{}).get("bpw","N/A")}')
print(f'  fingerprint : {stats.get("fingerprint","N/A")}')
print('\n✓ Cell 6 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Verify .axm proof ledger  (~10 s)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

REPO     = Path('/content/axiom')
AXM_PATH = Path('/content/gemma4_31b_srd4.axm')
assert AXM_PATH.is_file(), '.axm not found — run Cell 6 first'

result = subprocess.run(
    [sys.executable, 'axm_cli.py', 'verify', str(AXM_PATH)],
    cwd=REPO, capture_output=True, text=True,
)
try:
    output = json.loads(result.stdout)
except Exception:
    output = {'verified': False, 'raw': result.stdout + result.stderr}
print(json.dumps(output, indent=2))

if not output.get('verified'):
    raise RuntimeError('✗ Verification FAILED — container may be tampered')

FINGERPRINT = output.get('fingerprint', '')
print(f'\n✓ Verified  ({output.get("proofs_checked","?")} proofs)')
print(f'  fingerprint : {FINGERPRINT}')
print('\n✓ Cell 7 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Extract .axm → FP16 → GGUF Q4_K_M  (~20–35 min)
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

REPO      = Path('/content/axiom')
AXM_PATH  = Path('/content/gemma4_31b_srd4.axm')
GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
LLAMA_DIR = Path('/content/llama.cpp')

assert AXM_PATH.is_file(), '.axm not found — run Cell 6 first'
assert (LLAMA_DIR/'build/bin/llama-quantize').is_file(), 'llama-quantize missing — re-run Cell 1'

print(f'Extracting → {GGUF_PATH.name}  (Q4_K_M, est ~19 GB) ...')
t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'research.quant.axm_to_gguf',
     '--container', str(AXM_PATH),
     '--gguf-out',  str(GGUF_PATH),
     '--llamacpp',  str(LLAMA_DIR),
     '--quant',     'Q4_K_M',
     '--device',    'cuda'],
    cwd=REPO, check=True,
)
elapsed = time.time() - t0
size_gb = GGUF_PATH.stat().st_size / 1024**3
print(f'\n✓ {elapsed/60:.1f} min  →  {size_gb:.2f} GB  ({GGUF_PATH.name})')
print('\n✓ Cell 8 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Axiom MET metadata sidecar
# ══════════════════════════════════════════════════════════════════════════════
import json, time
from pathlib import Path

GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
MERGED    = Path('/content/gemma4_31b_axiom_merged')
assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 8 first'

try:
    FINGERPRINT = FINGERPRINT
except NameError:
    FINGERPRINT = ''

# Read actual architecture from merged checkpoint config
cfg_path = MERGED / 'config.json'
if cfg_path.is_file():
    cfg    = json.loads(cfg_path.read_text())
    HIDDEN = cfg.get('hidden_size', 5120)
    LAYERS = cfg.get('num_hidden_layers', 46)
    VOCAB  = cfg.get('vocab_size', 262144)
else:
    HIDDEN, LAYERS, VOCAB = 5120, 46, 262144
    print('[warn] config.json not found — using 31B defaults')

EMBEDDING_MB = round(VOCAB * HIDDEN * 2 / 1024**2, 1)

def _split(n, fracs):
    cuts, lo = [], 0
    for f in fracs:
        hi = min(lo + max(1, round(n * f)), n)
        cuts.append((lo, hi - 1))
        lo = hi
    cuts[-1] = (cuts[-1][0], n - 1)
    return cuts

RANGES     = _split(LAYERS, [0.20, 0.20, 0.37, 0.23])
SLOT_NAMES = ['early', 'factual', 'reasoning', 'governance']
transformer_params = 31e9 - VOCAB * HIDDEN
SLOT_RANGES = {}
for name, (lo, hi), frac in zip(SLOT_NAMES, RANGES, [0.20, 0.20, 0.37, 0.23]):
    mb = round(transformer_params * frac * 4.5 / 8 / 1024**2)
    SLOT_RANGES[name] = {'layers': f'{lo}-{hi}', 'first_layer': lo, 'last_layer': hi,
                         'mb': mb, 'precision': 'Q4_K_M'}

HYDRATION_POLICY = {
    'INFORM':    ['early'],
    'CLARIFY':   ['early', 'governance'],
    'REFUSE':    ['early', 'governance'],
    'UNCERTAIN': ['early', 'governance'],
    'HARM':      ['early', 'factual', 'reasoning', 'governance'],
    'DECEIVE':   ['early', 'factual', 'reasoning', 'governance'],
}

chunk_map = {}
for slot, info in SLOT_RANGES.items():
    for idx in range(info['first_layer'], info['last_layer'] + 1):
        chunk_map[str(idx)] = slot

slot_mb = {s: SLOT_RANGES[s]['mb'] for s in SLOT_NAMES}
STORAGE_MBS = 3000   # NVMe (31B is a server model, not phone)
intent_ram = {}
for intent, chunks in HYDRATION_POLICY.items():
    xfmr_mb = sum(slot_mb[c] for c in chunks)
    intent_ram[intent] = {
        'chunks': chunks, 'transformer_mb': xfmr_mb,
        'total_mb': EMBEDDING_MB + xfmr_mb,
        'nvme_load_ms': round(xfmr_mb / STORAGE_MBS * 1000, 1),
    }

meta = {
    'axiom_version': '1.4',
    'generated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model_id': 'google/gemma-4-31B-it-assistant (Axiom fine-tune)',
    'gguf_path': str(GGUF_PATH),
    'gguf_mb': round(GGUF_PATH.stat().st_size / 1024**2),
    'fingerprint': FINGERPRINT,
    'architecture': {'hidden_size': HIDDEN, 'num_layers': LAYERS, 'vocab_size': VOCAB},
    'fine_tuned_on': 'Axiom training corpus (behavioral + metric-targeted)',
    'embedding_slot': {
        'mb': EMBEDDING_MB, 'precision': 'F16', 'always_pinned': True,
        'note': f'{VOCAB:,}-token vocab — {EMBEDDING_MB:.0f} MB F16 (server class)'
    },
    'transformer_chunks': SLOT_RANGES,
    'chunk_map': chunk_map,
    'hydration_policy': HYDRATION_POLICY,
    'intent_ram_budget': intent_ram,
    'storage_speed_mbs': STORAGE_MBS,
    'between_met_floor_mb': EMBEDDING_MB,
    'peak_harm_mb': EMBEDDING_MB + sum(slot_mb.values()),
    'deployment_note': '31B is a server/workstation model — recommended: NVMe storage, 24+ GB VRAM'
}

sidecar = GGUF_PATH.with_suffix('.axiom_meta.json')
sidecar.write_text(json.dumps(meta, indent=2) + '\n')
print(f'✓ Sidecar → {sidecar}')

print(f'\n  Architecture : {LAYERS} layers · hidden {HIDDEN} · vocab {VOCAB:,}')
print(f'  Embedding    : {EMBEDDING_MB:.0f} MB F16  (always pinned)')
print()
print(f'  {"Slot":<14}  {"Layers":<10}  {"MB":>7}  Always?')
print('  ' + '─' * 42)
print(f'  {"embedding":<14}  {"—embed—":<10}  {EMBEDDING_MB:>7.0f}  ✓ pinned')
for slot, info in SLOT_RANGES.items():
    print(f'  {slot:<14}  {info["layers"]:<10}  {info["mb"]:>7}  on demand')
print()
print(f'  {"Intent":<10}  {"Total MB":>9}  {"NVMe ms":>8}')
print('  ' + '─' * 34)
for intent, info in intent_ram.items():
    print(f'  {intent:<10}  {info["total_mb"]:>9.0f}  {info["nvme_load_ms"]:>6.1f}ms')
print('\n✓ Cell 9 complete')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Smoke test + memory dashboard
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, time
from pathlib import Path

GGUF_PATH = Path('/content/gemma4_31b_q4km.gguf')
LLAMA_CLI = Path('/content/llama.cpp/build/bin/llama-cli')
sidecar   = GGUF_PATH.with_suffix('.axiom_meta.json')

assert GGUF_PATH.is_file(), 'GGUF not found — run Cell 8 first'
assert LLAMA_CLI.is_file(), 'llama-cli not found — re-run Cell 1'

# Gemma 4 chat format
PROMPT = ('<start_of_turn>user\n'
          'Classify this request using the Axiom verdict schema: '
          '"Delete all user records immediately."<end_of_turn>\n'
          '<start_of_turn>model\n')

print('Smoke test (32 tokens) ...')
t0 = time.time()
result = subprocess.run(
    [str(LLAMA_CLI),'-m',str(GGUF_PATH),'-p',PROMPT,
     '-n','32','--temp','0.0','-ngl','99','--log-disable'],
    capture_output=True, text=True, timeout=180,
)
elapsed = time.time()-t0
if result.returncode != 0:
    print(f'✗ failed:\n{result.stderr[-400:]}')
else:
    print(f'  Output : {repr(result.stdout.strip()[-150:])}')
    print(f'  Time   : {elapsed:.1f}s')
    print('  ✓ GGUF loads and generates')

# Memory dashboard
if sidecar.is_file():
    meta   = json.loads(sidecar.read_text())
    EMB_MB = meta['embedding_slot']['mb']
    PEAK   = meta['peak_harm_mb']
    GGUF_MB= meta['gguf_mb']
    ir     = meta.get('intent_ram_budget',{})
    W = 28
    def bar(mb): n=round(min(mb/PEAK,1)*W); return '█'*n+'░'*(W-n)

    print()
    print('═'*70)
    print('  AXIOM MET HYDRATION — Gemma 4 31B (Axiom fine-tuned)')
    print('─'*70)
    arch = meta.get('architecture',{})
    print(f'  {arch.get("num_layers","?")} layers · hidden {arch.get("hidden_size","?")} · '
          f'vocab {arch.get("vocab_size",0):,}')
    print(f'  GGUF : {GGUF_MB/1024:.1f} GB  |  Fingerprint: {meta.get("fingerprint") or "(not set)"}')
    print(f'  Fine-tuned on: {meta.get("fine_tuned_on","")}')
    print()
    print(f'  {"Static GGUF":<32}  {GGUF_MB:>7} MB  {bar(GGUF_MB)}')
    print(f'  {"Embedding floor":<32}  {EMB_MB:>7.0f} MB  {bar(EMB_MB)}')
    print(f'  {"Peak HARM":<32}  {PEAK:>7.0f} MB  {bar(PEAK)}')
    print()
    for intent, info in ir.items():
        tmb = info['total_mb']
        print(f'  {intent:<10}  {tmb:>7.0f} MB  {bar(tmb)}')
    print()
    print(f'  Note: 31B embedding = {EMB_MB:.0f} MB — server class, not phone.')
    print(f'  Deploy on: RunPod A100, Lambda Labs, local RTX 3090/4090 (24 GB).')
    print('═'*70)

print('\n✓ Pipeline complete.')
print(f'  GGUF   : {GGUF_PATH}')
print(f'  Sidecar: {sidecar}')
print()
print('Download:')
print('  from google.colab import files')
print(f'  # GGUF is ~19 GB — use Drive instead:')
print(f'  import shutil')
print(f'  shutil.copy("{GGUF_PATH}", "/content/drive/MyDrive/gemma4_31b_q4km.gguf")')
print(f'  shutil.copy("{sidecar}",   "/content/drive/MyDrive/gemma4_31b_q4km.axiom_meta.json")')